# 🚀 Huấn luyện PhoBERT v2 (`vinai/phobert-base-v2`) Tối ưu Tốc độ & Hiển thị Tiến độ

Notebook này được cấu hình **Ẩn các cảnh báo thừa**, **Tăng Batch Size = 32** để giảm thời gian train xuống 5-10 lần, và **Cập nhật Bảng Loss & Tiến độ liên tục** theo từng bước.

In [ ]:
# 1. Cài đặt các thư viện cần thiết & Tắt cảnh báo thừa
!pip install -q transformers datasets accelerate scikit-learn pandas pyarrow pyvi sentencepiece

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 2. Import các thư viện
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    RobertaTokenizerFast,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    logging as hf_logging
)

# Ẩn các log/warning không cần thiết từ HuggingFace
hf_logging.set_verbosity_error()

In [ ]:
# 3. Định nghĩa các hàm tiện ích xử lý dữ liệu
def find_char_span(context: str, answer: str):
    if not answer or not context:
        return -1, -1
    c_chars = [(char, i) for i, char in enumerate(context) if char not in (' ', '_')]
    a_chars = [char for char in answer if char not in (' ', '_')]
    c_str = ''.join([x[0] for x in c_chars])
    a_str = ''.join(a_chars)
    idx = c_str.find(a_str)
    if idx == -1:
        return -1, -1
    start_char_idx = c_chars[idx][1]
    end_char_idx = c_chars[idx + len(a_str) - 1][1] + 1
    return start_char_idx, end_char_idx

def load_qa_dataset(file_path: str) -> Dataset:
    df = pd.read_parquet(file_path)
    if "context_segmented" in df.columns:
        df = df.drop(columns=["context", "question", "answer_text"], errors="ignore")
        df = df.rename(columns={
            "context_segmented": "context",
            "question_segmented": "question",
            "answer_text_segmented": "answer_text"
        })
    data_dict = {
        "id": df["id"].astype(str).tolist(),
        "context": df["context"].astype(str).tolist(),
        "question": df["question"].astype(str).tolist(),
        "answer_text": df["answer_text"].fillna("").astype(str).tolist(),
        "answer_start": df["answer_start"].fillna(-1).astype(int).tolist(),
    }
    return Dataset.from_dict(data_dict)

def prepare_train_features(examples, tokenizer, max_seq_len=256, doc_stride=32):
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_seq_len,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        
        raw_context = examples["context"][sample_index]
        raw_answer = examples["answer_text"][sample_index]
        raw_start = examples["answer_start"][sample_index]

        if raw_start == -1 or not raw_answer.strip():
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        start_char, end_char = find_char_span(raw_context, raw_answer)
        if start_char == -1 or end_char == -1:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
            continue

        token_start_index = 0
        while token_start_index < len(sequence_ids) and sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while token_end_index >= 0 and sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if token_start_index > token_end_index or not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                token_start_index += 1
            tokenized_examples["start_positions"].append(token_start_index - 1)

            while token_end_index >= 0 and offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples

In [ ]:
# 4. Đọc dữ liệu (Tự động tìm đường dẫn trên Kaggle hoặc Colab)
import os
train_path = "viquad_train_segmented.parquet"
val_path = "viquad_val_segmented.parquet"

if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if "train" in f and f.endswith(".parquet"):
                train_path = os.path.join(root, f)
            elif "val" in f and f.endswith(".parquet"):
                val_path = os.path.join(root, f)

print(f"Train path: {train_path}")
print(f"Val path:   {val_path}")

train_dataset = load_qa_dataset(train_path)
val_dataset = load_qa_dataset(val_path)
print(f"Tập Train: {len(train_dataset)} mẫu, Tập Val: {len(val_dataset)} mẫu")

In [ ]:
# 5. Tải Tokenizer và tiền xử lý dữ liệu
model_name = "vinai/phobert-base-v2"
print(f"Đang tải RobertaTokenizerFast từ {model_name}...")
tokenizer = RobertaTokenizerFast.from_pretrained(model_name)

max_seq_len = 256
doc_stride = 32

print("Đang tiền xử lý (Tokenize) dữ liệu...")
tokenized_train = train_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, max_seq_len, doc_stride),
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_val = val_dataset.map(
    lambda x: prepare_train_features(x, tokenizer, max_seq_len, doc_stride),
    batched=True,
    remove_columns=train_dataset.column_names
)
print(f"Số lượng features sau Tokenize - Train: {len(tokenized_train)}, Val: {len(tokenized_val)}")

In [ ]:
# 6. Huấn luyện (Tăng tốc & Hiện tiến độ liên tục vào bảng)
print(f"Đang khởi tạo mô hình PhoBERT v2...")
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

output_dir = "./results_phobert"

eval_key = "eval_strategy" if hasattr(TrainingArguments("./tmp"), "eval_strategy") else "evaluation_strategy"
kwargs_args = {
    "output_dir": output_dir,
    eval_key: "steps",
    "eval_steps": 500,              # Tính Loss tập Val mỗi 500 bước để hiện vào bảng
    "learning_rate": 3e-5,
    "per_device_train_batch_size": 32, # Tăng Batch Size lên 32 để chạy siêu nhanh (giảm tổng số bước xuống 3,000)
    "per_device_eval_batch_size": 32,
    "num_train_epochs": 2,            # 2 Epochs là đủ tối ưu xuất sắc
    "weight_decay": 0.01,
    "save_total_limit": 1,
    "logging_steps": 50,              # Cập nhật Loss tập Train mỗi 50 bước vào bảng
    "save_strategy": "steps",
    "save_steps": 500,
    "load_best_model_at_end": True,
    "metric_for_best_model": "loss",
    "greater_is_better": False,
    "report_to": "none",
    "fp16": torch.cuda.is_available()  # Bật mixed-precision fp16 nếu dùng GPU T4 để tăng tốc gấp 2 lần
}
training_args = TrainingArguments(**kwargs_args)

try:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        processing_class=tokenizer,
    )
except TypeError:
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
    )

print("🚀 Bắt đầu quá trình huấn luyện PhoBERT v2...")
trainer.train()
print("✅ Huấn luyện hoàn tất!")

In [ ]:
# 7. Lưu mô hình & nén zip
final_model_dir = "./vinai_phobert-base-v2"
print(f"Đang lưu mô hình tốt nhất vào {final_model_dir}...")
trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

!zip -r vinai_phobert-base-v2.zip {final_model_dir}
print("🎉 Đã nén xong! Hãy tải tệp 'vinai_phobert-base-v2.zip' về máy của bạn.")